# Veritas PoC on Google Colab

Runs the full fast-vs-slow causal-scoring pipeline on a Colab GPU (much faster than local CPU).

**Before running:** set the runtime to a GPU via `Runtime > Change runtime type > T4 GPU`.

You will also need a Hugging Face token with access to `meta-llama/Llama-3.2-1B-Instruct`
(accept the license at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct first).

In [ ]:
# 1. Confirm a GPU is attached
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2. Clone (or update) the repo and cd into it
import os

if os.path.basename(os.getcwd()) != 'algoverse-PoC':
    if not os.path.isdir('algoverse-PoC'):
        !git clone https://github.com/abdelmagid07/algoverse-PoC.git
    %cd algoverse-PoC
!git pull --ff-only

In [ ]:
# 3. Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# 4. Authenticate with Hugging Face (paste a token with Llama-3.2 access)
from getpass import getpass
from huggingface_hub import login

login(getpass('Enter your Hugging Face token: '))

In [ ]:
# 5. Phase 1: collect trajectories + cache activations (the slowest step)
!python -m agent.runner

In [ ]:
# 6. Phases 2-4: fast scores, slow scores, correlation, plot, summary
!python -m interp.attribution_patch
!python -m interp.ground_truth_patch
!python -m analysis.correlate
!python -m analysis.visualize
!python -m analysis.summarize

In [ ]:
# 7. Show the results inline
import json
from IPython.display import Image, Markdown, display

print(json.dumps(json.load(open('results/correlation.json')), indent=2))
display(Image('results/importance_by_step.png'))
display(Markdown(open('results/poc_summary.md').read()))